# Data Collection

Downloads OHLCV stock data for the portfolio universe via yfinance and saves
raw CSV files used by all subsequent notebooks.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import yfinance as yf

# Ensure project root is on the path so src imports work from any CWD
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JNJ', 'UNH', 'PFE', 'JPM', 'V', 'AMZN', 'WMT', 'XOM']
start = '2020-01-01'
end   = '2024-12-30'

In [ ]:
print('Downloading data...')
raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
print(f'Shape: {raw.shape}')

In [ ]:
open_p  = raw['Open']
high    = raw['High']
low     = raw['Low']
prices  = raw['Close']
volumes = raw['Volume']

In [ ]:
# Clean: forward-fill then back-fill; drop any column still fully NaN
prices  = prices.ffill().bfill().dropna(axis=1, how='all')
volumes = volumes.loc[prices.index, prices.columns]

In [ ]:
returns     = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

In [ ]:
os.makedirs('../data/raw', exist_ok=True)

ohlcv = pd.concat({'Open': open_p, 'High': high, 'Low': low,
                   'Close': prices, 'Volume': volumes}, axis=1)
ohlcv.to_csv('../data/raw/stock_data.csv')
returns.to_csv('../data/raw/stock_returns.csv')
log_returns.to_csv('../data/raw/stock_log_returns.csv')
print('Data saved to data/raw/')

In [ ]:
print(f'Tickers      : {list(prices.columns)}')
print(f'Date range   : {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Trading days : {len(prices)}')
print()
print(prices.tail(3))